# Tomato Leaf Disease Detection - Model Training
MobileNetV3-Large transfer learning -> TFLite for Raspberry Pi.

**Before you start (Colab):** Runtime -> Change runtime type -> **T4 GPU**.

Dataset zip must contain a top-level `dataset/` folder:
```
dataset/train/<class>/*.jpg    dataset/val/<class>/*.jpg    dataset/test/<class>/*.jpg
```
Build it with `prepare_dataset.py`, zip it, upload to Google Drive, then run the cells top to bottom.

In [ ]:
import os, zipfile, numpy as np, tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
# ===================== SETTINGS =====================
ZIP_PATH    = "/content/drive/MyDrive/leaf_dataset.zip"   # your uploaded zip
DATA_DIR    = "/content/dataset"                          # where it is unzipped
OUT_DIR     = "/content/drive/MyDrive/leaf_outputs"       # models/plots saved here
IMG_SIZE    = (224, 224)
BATCH       = 32
HEAD_EPOCHS = 10      # train new classifier layer
FINE_EPOCHS = 12      # fine-tune top of MobileNetV3
FINE_TUNE_LAYERS = 30
# ====================================================

In [ ]:
# Mount Drive + unzip (Colab). For local Jupyter: just set DATA_DIR to your dataset folder and skip this cell.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

os.makedirs(OUT_DIR, exist_ok=True)
if IN_COLAB and not os.path.isdir(DATA_DIR):
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall("/content/")
print("Found:", os.listdir(DATA_DIR))

## Load data

In [ ]:
train_raw = keras.utils.image_dataset_from_directory(f"{DATA_DIR}/train", image_size=IMG_SIZE, batch_size=BATCH, seed=42)
val_raw   = keras.utils.image_dataset_from_directory(f"{DATA_DIR}/val",   image_size=IMG_SIZE, batch_size=BATCH, shuffle=False)
test_raw  = keras.utils.image_dataset_from_directory(f"{DATA_DIR}/test",  image_size=IMG_SIZE, batch_size=BATCH, shuffle=False)

class_names = train_raw.class_names
assert class_names == val_raw.class_names == test_raw.class_names, "train/val/test class folders must match!"
num_classes = len(class_names)
print(num_classes, "classes:", class_names)

counts = {c: len(os.listdir(f"{DATA_DIR}/train/{c}")) for c in class_names}
for c, n in counts.items():
    print(f"{c:35s} {n}")

# class weights to handle imbalance
total = sum(counts.values())
class_weight = {i: total / (num_classes * counts[c]) for i, c in enumerate(class_names)}

# augmentation applied only to training images (input stays 0-255)
aug = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomBrightness(0.2),
    layers.RandomContrast(0.2),
])
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_raw.map(lambda x, y: (aug(x, training=True), y), num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds   = val_raw.prefetch(AUTOTUNE)
test_ds  = test_raw.prefetch(AUTOTUNE)

## Build model
Keras MobileNetV3 has preprocessing **built in** and expects raw 0-255 pixels. Do **not** divide by 255 (neither here nor in the Pi app).

In [ ]:
base = keras.applications.MobileNetV3Large(input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet")
base.trainable = False

inputs = keras.Input(IMG_SIZE + (3,))
x = base(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)
model = keras.Model(inputs, outputs)
model.summary()

## Stage 1 - train the new classifier head

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
cb1 = [keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True),
       keras.callbacks.ModelCheckpoint(f"{OUT_DIR}/head_best.keras", save_best_only=True)]
h1 = model.fit(train_ds, validation_data=val_ds, epochs=HEAD_EPOCHS, class_weight=class_weight, callbacks=cb1)

## Stage 2 - fine-tune top layers (small learning rate)

In [ ]:
base.trainable = True
for layer in base.layers[:-FINE_TUNE_LAYERS]:
    layer.trainable = False

model.compile(optimizer=keras.optimizers.Adam(1e-5),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
cb2 = [keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True),
       keras.callbacks.ModelCheckpoint(f"{OUT_DIR}/final_best.keras", save_best_only=True)]
h2 = model.fit(train_ds, validation_data=val_ds, epochs=FINE_EPOCHS, class_weight=class_weight, callbacks=cb2)
model.save(f"{OUT_DIR}/leaf_model.keras")

In [ ]:
acc = h1.history["accuracy"] + h2.history["accuracy"]
vacc = h1.history["val_accuracy"] + h2.history["val_accuracy"]
plt.figure(figsize=(7,4)); plt.plot(acc, label="train"); plt.plot(vacc, label="val")
plt.axvline(len(h1.history["accuracy"]) - 1, ls="--", c="gray"); plt.legend(); plt.title("Accuracy")
plt.xlabel("epoch"); plt.savefig(f"{OUT_DIR}/accuracy_curve.png", dpi=150, bbox_inches="tight"); plt.show()

## Evaluate on the TEST set
Look at per-class recall and the confusion matrix, not only overall accuracy.
If test accuracy is great but real webcam photos fail, add more of your own webcam images and retrain.

In [ ]:
y_true = np.concatenate([y.numpy() for _, y in test_raw])
y_pred = np.argmax(model.predict(test_ds), axis=1)

report = classification_report(y_true, y_pred, target_names=class_names, digits=3)
print(report)
open(f"{OUT_DIR}/classification_report.txt", "w").write(report)

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(num_classes)); ax.set_yticks(range(num_classes))
ax.set_xticklabels(class_names, rotation=90); ax.set_yticklabels(class_names)
for i in range(num_classes):
    for j in range(num_classes):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max()/2 else "black", fontsize=8)
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Confusion matrix (test)")
plt.savefig(f"{OUT_DIR}/confusion_matrix.png", dpi=150, bbox_inches="tight"); plt.show()

## Choose the confidence threshold
For each threshold: how many predictions are accepted (coverage) and how accurate are those?
Pick one where accuracy is high enough and coverage is still acceptable, then set `CONF_THRESHOLD` in `app.py`.

In [ ]:
val_probs = model.predict(val_ds)
val_true = np.concatenate([y.numpy() for _, y in val_raw])
conf = val_probs.max(axis=1); pred = val_probs.argmax(axis=1); correct = pred == val_true

print("threshold | coverage | accuracy on accepted")
for t in [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
    m = conf >= t
    a = correct[m].mean() if m.any() else float("nan")
    print(f"   {t:.2f}   |  {m.mean():6.1%}  |  {a:6.1%}")

## Export for Raspberry Pi (labels.txt + model.tflite)

In [ ]:
with open(f"{OUT_DIR}/labels.txt", "w") as f:
    f.write("\n".join(class_names))

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_model = converter.convert()
TFLITE_PATH = f"{OUT_DIR}/model.tflite"
open(TFLITE_PATH, "wb").write(tflite_model)
print("model.tflite size: %.1f MB" % (len(tflite_model) / 1e6))
print("labels order:", class_names)

## Verify the TFLite model matches the Keras model

In [ ]:
interp = tf.lite.Interpreter(model_path=TFLITE_PATH)
interp.allocate_tensors()
inp, out = interp.get_input_details()[0], interp.get_output_details()[0]
print("input:", inp["shape"], inp["dtype"])

agree = n = 0
for x, y in test_raw.unbatch().take(100):
    arr = x.numpy().astype("float32")[None]            # raw 0-255, same as the Pi app
    k = int(np.argmax(model.predict(arr, verbose=0)))
    interp.set_tensor(inp["index"], arr); interp.invoke()
    t = int(np.argmax(interp.get_tensor(out["index"])[0]))
    agree += (k == t); n += 1
print(f"Keras vs TFLite agreement: {agree}/{n}")

## Download

In [ ]:
if IN_COLAB:
    from google.colab import files
    files.download(TFLITE_PATH)
    files.download(f"{OUT_DIR}/labels.txt")
    files.download(f"{OUT_DIR}/confusion_matrix.png")
    files.download(f"{OUT_DIR}/classification_report.txt")
else:
    print("Files are in", OUT_DIR)